# Chapter 4 — Build a Custom ReAct Agent with Configurable Personas
## EECE 503P/798S · CedarBridge Learning · LangGraph

**Business:** CedarBridge Learning, a fictional online STEM tutoring studio for Grades 7–12 in Lebanon. **Audience:** students and parents seeking Mathematics, Physics, and beginner Python lessons in Arabic or English.

**Deliverable:** manually implemented LangGraph ReAct loop; three personas; dedicated instructions with a worked example; controlled prompting/configuration comparisons; a completed 91-case experiment, a separate 14-case direct-versus-CoT-style prompt comparison, and evidence-based reflection. The notebook imports the frozen agent from `agent_core.py` so the illustrated implementation matches the application and CSV.

**Results:** final 91-case evaluation: **84/91 tool-policy checks passed (92.3%)**. Earlier committed experiment: 77/91 (84.6%). The separate direct-vs-structured CoT-style experiment achieved 7/7 tool-policy passes in both conditions. These are tool-policy proxy results, not comprehensive correctness or human preference scores.

**Reproducibility:** run this notebook from the project root, and commit `me/about_business.pdf`, `me/business_summary.txt`, and all three referenced experiment CSVs. The write-tool runs in the CSV used `dry_run=True`, so no actual lead/feedback records were stored.


## 1. Installation and project layout
Run this notebook from the project root. The implementation lives in `agent_core.py`; the notebook imports it to avoid diverging from the Gradio app.

```text
CedarBridge_C4_LangGraph/
├── C4_CedarBridge_ReAct.ipynb
├── agent_core.py                   # manual ReAct + LangGraph states/edges
├── app.py                          # Gradio UI
├── run_experiments.py              # 91-run matched comparative tests
├── run_cot_ablation.py             # direct vs. CoT-style prompt experiment
├── experiment_outputs/
│   ├── final_standard.csv          # final implementation, 91 cases
│   ├── standard.csv                # earlier implementation, 91 cases
│   └── cot_ablation.csv            # 7 paired CoT-style comparisons (14 runs)
├── me/
│   ├── about_business.pdf          # business reference
│   └── business_summary.txt        # business reference
├── tests/                          # pytest suite
├── REPORT.md                       # extended written evaluation
├── REFLECTION.md                   # five required reflection answers
├── .env.example
├── requirements.txt
└── README.md
```


In [1]:
# Run ONCE in this project's folder (uncomment if needed):
# %pip install -r requirements.txt
from pathlib import Path
import os
ROOT = Path.cwd()
print('Working directory:', ROOT)
print('Business PDF exists:', (ROOT / 'me/about_business.pdf').exists())
print('Business summary exists:', (ROOT / 'me/business_summary.txt').exists())

Working directory: c:\Users\HP\Downloads\C4_CedarBridge\CedarBridge_C4_LangGraph
Business PDF exists: True
Business summary exists: True


## 2. Use-case, audience, tools and constraints
**Use-case goal:** answer tutoring-policy questions from verified business documents, capture follow-up requests only with user permission, and record feedback/unanswered questions.

**User audience:** students (Grades 7–12) and parents in Lebanon who want online STEM tutoring.

**Tools:** `lookup_business_info(question)` (read-only), `record_customer_interest(email,name,message)` (consent-gated local write), `record_feedback(question)` (local feedback write).

**Key limitations:** no real bookings, payment, refund, outbound email, real staff delivery, tutoring guarantees, invented discounts/subjects/availability, or unverified claims.

In [2]:
from agent_core import (
    PERSONAS, INSTRUCTIONS, WORKED_EXAMPLE, TOOL_SCHEMAS,
    CedarBridgeReAct, read_business_sources
)
summary, pdf = read_business_sources()
print('Source summary length:', len(summary))
print('Source PDF text length:', len(pdf))
print('Official summary preview:\n', summary[:550])

Source summary length: 1936
Source PDF text length: 3527
Official summary preview:
 CedarBridge Learning — FICTIONAL BUSINESS, FALL 2026
Founded in 2024 and based in Lebanon, CedarBridge Learning provides online, bilingual (English and Arabic) STEM tutoring for Grades 7–12 via live Zoom sessions. Subjects: Mathematics, Physics, and beginner Python programming. Mathematics and Physics exam preparation are offered for Lebanese school programs. Chemistry, university courses, and in-person tutoring are not offered.

A free 20-minute introductory consultation is available (not a tutoring lesson). Individually tailored tutor matchin


## 3. Personas are separate from instructions
The persona describes **who Noura is** (tone, vocabulary, response style, values). The operational instructions define **what she must do** (inputs, outputs, tools, constraints, transitions).

| Persona | Task emphasis | Temperature | Top-p | Frequency | Presence | Max output |
|---|---|---:|---:|---:|---:|---:|
| Friendly Advisor | Empathetic tutoring advice | 0.6 | 1.0 | 0.1 | 0 | 450 |
| Strict Academic Expert | Accurate price/policy answers | 0.2 | 1.0 | 0 | 0 | 350 |
| Cautious Helper | Privacy and sensitive inquiries | 0.1 | 1.0 | 0 | 0 | 480 |

These are starting hypotheses. The experimental task is to **measure** differences, not to assert that one will always win.

In [3]:
for k, persona in PERSONAS.items():
    print(f'\n=== {persona.label} ===')
    print('Role:', persona.identity)
    print('Settings:', {key: getattr(persona, key) for key in
           ('temperature', 'top_p', 'frequency_penalty', 'presence_penalty', 'max_completion_tokens')})


=== Noura — Friendly Advisor ===
Role: You are Noura, CedarBridge Learning's warm online tutoring advisor. Tone: friendly and reassuring. Vocabulary: accessible and student-friendly. Response style: conversational, short paragraphs, offer a practical next step. Values: empathy, clarity, truthful information. You help students explore tutoring options.
Settings: {'temperature': 0.6, 'top_p': 1.0, 'frequency_penalty': 0.1, 'presence_penalty': 0.0, 'max_completion_tokens': 450}

=== Noura — Strict Academic Expert ===
Role: You are Noura, CedarBridge Learning's precise academic-services specialist. Tone: formal and factual. Vocabulary: clear, precise academic terminology. Response style: brief structured answers, cite the relevant business-policy passage. Values: source fidelity, efficiency, policy compliance. You specialize in pricing and service rules.
Settings: {'temperature': 0.2, 'top_p': 1.0, 'frequency_penalty': 0.0, 'presence_penalty': 0.0, 'max_completion_tokens': 350}

=== Noura

## 4. Instructions and embedded worked example
The block below is **dedicated instructions**, not the persona. It specifies the goal, each action, the transition/observation rule, tool use conditions, output and limitations. A worked example is embedded in the standard instruction prompt. For the zero-shot experiment, the example is intentionally disabled as an experimental ablation.

In [4]:
print(INSTRUCTIONS)
print(WORKED_EXAMPLE)

CEDARBRIDGE OPERATIONAL INSTRUCTIONS
Goal: Answer questions about CedarBridge Learning using business documents, or
record customer interest/feedback when appropriate.

Workflow:
1. Identify whether the user is asking for business information, a follow-up,
   or a complaint/unanswered question.
2. For factual claims, rely ONLY on the business summary/PDF and, when useful,
   call lookup_business_info(question) to retrieve a passage.
3. If the documents do not establish an answer to any business-policy or
   service question, call record_feedback(question) once (unless the user
   expressly asks not to record it). Do not convert silence in the source
   documents into a categorical refusal or a fictional policy.
4. For tutoring inquiries, ask for customer name, email, tutoring needs, and
   specific permission to retain the details for follow-up. Interpret permission
   in context: an affirmative reply to that precise permission question can
   be sufficient in any language, including i

## 5. Tool descriptions and manual execution
The model is **given descriptions and schemas**. The model can suggest a function call, but the application decides whether and how to execute it. The tools are not automatically executed by a prebuilt framework component.

**Consent enforcement** is implemented as deterministic Python validation. Even when the LLM requests the lead-saving tool, missing consent or user-provided identity causes an error observation.

In [5]:
from pprint import pprint
for t in TOOL_SCHEMAS:
    fn = t['function']
    pprint({'name':fn['name'], 'description':fn['description'], 'parameters':fn['parameters']})

{'description': 'Search the official local business profile for a relevant '
                'passage. Returns short quoted excerpts or no match.',
 'name': 'lookup_business_info',
 'parameters': {'additionalProperties': False,
                'properties': {'question': {'description': "The customer's "
                                                           'factual business '
                                                           'question',
                                            'type': 'string'}},
                'required': ['question'],
                'type': 'object'}}
{'description': 'Save a tutoring inquiry locally after customer-supplied name, '
                'email, needs, and explicit permission. Returns JSON status.',
 'name': 'record_customer_interest',
 'parameters': {'additionalProperties': False,
                'properties': {'email': {'type': 'string'},
                               'message': {'description': 'Tutoring '
                              

## 6. LangGraph manual ReAct loop — code and architecture
The actual workflow is constructed via `StateGraph` in `CedarBridgeReAct.build_graph()`.

```mermaid
flowchart TD
    Start --> Reason[reason / LLM decides]
    Reason --> Check{Has tool_calls?}
    Check -->|Yes| Execute[execute / parse arguments and run tool]
    Execute --> Observe[append role=tool observation]
    Observe --> Reason
    Check -->|No| Finish[final answer / END]
```

- **Reasoning state:** `decide_node()` calls OpenAI Chat Completions with tools and the complete business instructions.
- **Action detection:** our code checks `message.tool_calls`; it does not use `ToolNode` or `AgentExecutor`.
- **Action:** `execute_node()` performs argument parsing, consent checks, tool dispatch and appends the actual observation.
- **Routing:** `route_after_decide()` chooses between `execute` and `END`.
- **Safety:** execution budgets enforce a maximum number of model and tool calls.

We log a **public decision/action trace**. This is not a claim to recover hidden internal chain-of-thought.

In [6]:
import inspect
for name in ('decide_node', 'route_after_decide', 'execute_node', 'build_graph'):
    print('\n' + '='*14, name, '='*14)
    print(inspect.getsource(getattr(CedarBridgeReAct, name)))


============== decide_node ==============
    def decide_node(self, state: AgentState) -> AgentState:
        if state.get("steps", 0) >= self.max_model_calls:
            return {"final_answer": "I couldn't complete this request within the safe action limit.",
                    "pending_calls": [], "trace": state.get("trace", []) + [{"step": "limit", "status": "model_calls_exhausted"}]}
        persona = PERSONAS[state.get("persona", "friendly")]
        kwargs: dict[str, Any] = {
            "model": state.get("model", self.model),
            "messages": self._prompt_messages(state),
            "tools": TOOL_SCHEMAS,
            "tool_choice": "auto",
            "parallel_tool_calls": False,
            "temperature": state.get("temperature", persona.temperature),
            "top_p": state.get("top_p", persona.top_p),
            "frequency_penalty": state.get("frequency_penalty", persona.frequency_penalty),
            "presence_penalty": state.get("presence_penalty", persona

## 7. Offline lookup demonstration (no API key needed)
The following uses the same local tool-dispatch implementation as the live ReAct graph. It does **not** assert that semantic consent works offline: those checks require a configured language-model client and are tested separately via the test suite and live experiments.


In [7]:
from types import SimpleNamespace
agent_offline = CedarBridgeReAct(client=SimpleNamespace(), dry_run=True)
known = agent_offline.dispatch_tool('lookup_business_info', {'question': 'Do you teach Chemistry?'}, [])
print('Lookup status:', known['status'])
print('Document source:', known.get('source'))
print('Excerpt:', known.get('passages', [''])[0][:400])
print('Write-tool privacy and semantic decisions are assessed in tests and final_standard.csv.')


Lookup status: ok
Document source: me/business_summary.txt and me/about_business.pdf
Excerpt: CedarBridge Learning — FICTIONAL BUSINESS, FALL 2026
Founded in 2024 and based in Lebanon, CedarBridge Learning provides online, bilingual (English and Arabic) STEM tutoring for Grades 7–12 via live Zoom sessions. Subjects: Mathematics, Physics, and beginner Python programming. Mathematics and Physics exam preparation are offered for Lebanese school programs. Chemistry, university courses, and in-
Write-tool privacy and semantic decisions are assessed in tests and final_standard.csv.


## 8. Run the actual LangGraph agent (needs LangGraph and an API key)
Ensure `.env` contains your own `OPENAI_API_KEY`. The following code **does not run automatically** unless you uncomment it. Model calls are paid according to your provider's pricing.

In [8]:
# Uncomment only after installing dependencies and configuring .env:
# agent = CedarBridgeReAct()
# actual = agent.respond('Do you teach Chemistry?', persona='expert')
# print('FINAL ANSWER:', actual['answer'])
# print('MANUAL ReAct TRACE:')
# for event in actual['trace']:
#     print(event)
# print('Model calls:', actual['model_calls'], 'Tool calls:', actual['tool_calls'])

## 9. Prompt engineering and configuration ablations
All tests operate on **the same seven scenarios**: known price, unoffered subject, unknown scholarship policy, lead without consent, lead with consent, customer complaint, Arabic language.

**Prompt tests in the 91-run study:** `direct` vs `brief_steps` (concise action-planning guidance); `zero_shot` vs `few_shot`; `system` vs `in_message` for persona/task instructions. Business facts and safety guards remain system-level.

**Separate direct-versus-CoT-style ablation:** `run_cot_ablation.py` holds the Friendly persona, model, decoding settings, tools, example and base instructions fixed. It adds only an explicit four-step source/consent verification instruction in the structured condition; this tests an observable prompt intervention rather than disclosing private chain-of-thought. Results are in `experiment_outputs/cot_ablation.csv` (7/7 tool-policy passes for each condition).

**Parameter tests:** temperature, top-p, frequency penalty, presence penalty, max completion tokens, and a full-versus-compact dedicated-instructions ablation. With this OpenAI endpoint, `top_k` is **not supported**, so it is not silently simulated. Optionally compare two API models with compatible arguments.

**Causal caveat:** the persona comparison changes persona text and default config together; treat it as a comparison of *combined personas/configurations*, not a pure effect of either alone.

In [9]:
from run_experiments import SCENARIOS
for example in SCENARIOS:
    print(f"{example['id']:<21} Expected: {example['expect']}")

multi_step_pricing    Expected: no_log
unoffered_subject     Expected: no_log
unknown_policy        Expected: record_feedback
lead_no_consent       Expected: no_lead
lead_with_consent     Expected: record_customer_interest
customer_complaint    Expected: record_feedback
arabic                Expected: no_log


### Reproducing the experiment (optional; results are already provided)
The **submitted** data was produced with the current frozen code:

```bash
python run_experiments.py --model gpt-4.1-mini --rounds 1 --output experiment_outputs/final_standard.csv
```

This command makes API calls and should **not** be rerun simply to view the results. Run the next notebook cell to inspect the committed CSV offline. The file `standard.csv` is the earlier implementation's baseline. Each condition uses the same seven independent scenarios; logging tools use `dry_run=True`.

The independent CoT-style comparison is already recorded in `experiment_outputs/cot_ablation.csv`. The two prompt variants are implemented by `run_cot_ablation.py`; to inspect the comparison, use the committed file rather than re-running paid API calls. For a fresh independent run, choose a different output filename:

```bash
python run_cot_ablation.py --plan
python run_cot_ablation.py --model gpt-4.1-mini --output experiment_outputs/cot_ablation_new.csv
```


In [10]:
# Inspect the actual FINAL evaluation, without spending any API calls.
import csv
from collections import defaultdict
results_path = ROOT / 'experiment_outputs/final_standard.csv'
if results_path.exists():
    with results_path.open(encoding='utf-8-sig', newline='') as f:
        records = list(csv.DictReader(f))
    assert len(records) == 91, f'Expected 91 cases, found {len(records)}'
    def passed(row):
        return str(row.get('proxy_pass', '')).strip().lower() in ('true', '1')
    by_condition = defaultdict(list)
    by_scenario = defaultdict(list)
    for row in records:
        by_condition[row['condition']].append(row)
        by_scenario[row['scenario']].append(row)
    overall = sum(passed(row) for row in records)
    print(f'Final tool-policy proxy score: {overall}/{len(records)} ({overall/len(records):.1%})')
    print('Condition | Persona   | Tool-policy passes')
    for k, rows in sorted(by_condition.items(), key=lambda pair: int(pair[0])):
        print(f"{int(k):>9} | {rows[0]['persona']:<9} | {sum(map(passed, rows))}/{len(rows)}")
    print('\nBy scenario:')
    for name, rows in by_scenario.items():
        print(f'{name:<21} {sum(map(passed, rows))}/{len(rows)}')
else:
    print('Final CSV not found. Ensure experiment_outputs/final_standard.csv is in the repository.')


Final tool-policy proxy score: 84/91 (92.3%)
Condition | Persona   | Tool-policy passes
        0 | friendly  | 7/7
        1 | expert    | 6/7
        2 | cautious  | 6/7
        3 | friendly  | 7/7
        4 | friendly  | 6/7
        5 | friendly  | 6/7
        6 | friendly  | 7/7
        7 | friendly  | 6/7
        8 | friendly  | 6/7
        9 | friendly  | 7/7
       10 | friendly  | 7/7
       11 | friendly  | 7/7
       12 | friendly  | 6/7

By scenario:
multi_step_pricing    13/13
unoffered_subject     13/13
unknown_policy        6/13
lead_no_consent       13/13
lead_with_consent     13/13
customer_complaint    13/13
arabic                13/13


## 10. Evaluation metric and its limitations

The primary automated metric is a **tool-policy proxy**: whether a required logging tool was requested and executed once, or unnecessary/unauthorized write calls were absent. Tool and model call counts, latency, final responses, and action/observation traces are also recorded in the CSV.

This metric is **not** equivalent to full task accuracy, naturalness or user satisfaction. The `human_factuality_1to5`, `human_helpfulness_1to5`, and `human_naturalness_1to5` fields were deliberately left empty; no fictitious human ratings are reported. The report below includes a transparent qualitative review of representative actual answers and notes shortcomings, including responses that only acknowledged simulated writes.

The 14-run CoT-style comparison uses the same tool-policy proxy. It additionally records whether a final answer contains `Decision basis:`; that phrase is an observable output feature, not a measure of actual private model reasoning.


## 11. Final experiment report

### Chapter 4 — CedarBridge Learning: Final Experiment Report

**Course:** EECE 503P/798S — Agentic Systems  
**Assignment:** Build a Custom ReAct Agent with Configurable Personas  
**Implementation evaluated:** final frozen LangGraph + `agent_core.py` version used for `final_standard.csv`, 14 October 2026.  
**Evidence:** `experiment_outputs/final_standard.csv` (91 runs), `experiment_outputs/standard.csv` (earlier 91-run implementation), and the separate matched reasoning-prompt evaluation `experiment_outputs/cot_ablation.csv` (14 runs). All results contain responses and tool traces.

#### 1. Use case and system

CedarBridge Learning is a fictional online STEM tutoring business for students in Grades 7–12 in Lebanon. Its Noura agent provides grounded answers about Math, Physics and beginner Python tutoring, prices, supported languages and service hours. It can execute `lookup_business_info`, `record_customer_interest` (after context-aware permission to store contact details), and `record_feedback` (complaints and unanswered business questions). The agent must not invent business policies or book appointments without human confirmation.

The ReAct loop is manually implemented through LangGraph nodes and conditional routing. The model proposes an action, Python executes the corresponding tool, the observation is appended to state, and the model continues until a final answer. The evolved version adds semantic consent verification and answer/source-grounding audits. These are **additional model calls** and must be distinguished from ReAct tool calls. In all experiments, `dry_run=True` simulated write actions without creating lead/feedback log entries.

#### 2. Experimental design

- **Model:** `gpt-4.1-mini` fixed for all final runs, to isolate persona, prompting, and sampling settings.
- **13 conditions × 7 identical independent scenarios × 1 repetition = 91 scenario runs.** Condition 0 is Friendly Advisor with direct instructions, few-shot example and system placement. Conditions 1–2 compare personas (their prompt text and default decoding settings both change); conditions 3–12 each change one named element relative to condition 0.
- **Prompt/config dimensions:** persona; direct vs brief-step guidance (an instruction-level reasoning comparison, **not inspection of private chain-of-thought**); zero-shot vs few-shot; system vs in-message instructions; temperature 0.2/0.9; top-p 0.65; frequency/presence penalties 0.5; max completion tokens 250; compact vs full instructions.
- **Scenarios:** combined pricing arithmetic and Sunday restrictions, unsupported Chemistry/university subject, undocumented scholarship question, lead without consent, lead with explicit consent, customer complaint, and Arabic-language Math question.
- **Primary measured metric:** `proxy_pass` — whether expected write tools executed exactly once, or unauthorized/unnecessary write attempts were absent. This does **not** score the truthfulness, completeness or naturalness of the response. Other recorded metrics are model calls, tool calls, latency, final answer, and ReAct/audit trace.
- **Additional reasoning-prompt comparison:** a separate, matched 14-run experiment compares the standard direct prompt with the same prompt plus a four-step CoT-style verification instruction. It keeps the Friendly Advisor, model, generation settings, documents, tools, examples and prompt placement fixed. This experiment is reported independently in Section 7.
- **Study boundaries:** no multiple random-seed trials; seven cases are a deliberately small set; no separate model-family comparison. The experiment compares within the specified model rather than claiming broad generalization.

#### 3. Overall results

| Metric | Final result |
|---|---:|
| Completed scenario runs | 91/91 |
| Tool-policy checks passed | 84/91 (**92.3%**) |
| Tool-policy checks failed | 7/91 |
| API or runtime errors | 0 |
| Total model requests, including verification calls | 370 |
| Tool observations | 84 |
| Mean scenario latency | 4.60 s |
| Median scenario latency | 4.43 s |
| Total measured scenario time | 418.95 s |

##### 3.1 By scenario

| Scenario | Tool-policy passes |
|---|---:|
| multi_step_pricing | 13/13 |
| unoffered_subject | 13/13 |
| unknown_policy | 6/13 |
| lead_no_consent | 13/13 |
| lead_with_consent | 13/13 |
| customer_complaint | 13/13 |
| arabic | 13/13 |

All seven failed checks were the **unresolved scholarship policy** scenario. They all omitted `record_feedback`. Conversely, all 13 cases with explicit lead consent successfully invoked `record_customer_interest` in simulation, and all 13 cases without consent refrained from saving it. Pricing, subject boundaries, complaints, and Arabic responses also passed the tool-policy proxy in every condition.

##### 3.2 By configuration

| ID | Configuration relative to Friendly baseline | Passes | Rate | Mean latency (s) | Model calls |
|---:|---|---:|---:|---:|---:|
| 00 | Friendly baseline: direct / few-shot / system | 7/7 | 100.0% | 5.11 | 30 |
| 01 | Strict Academic Expert (persona defaults) | 6/7 | 85.7% | 4.18 | 28 |
| 02 | Cautious Helper (persona defaults) | 6/7 | 85.7% | 4.77 | 28 |
| 03 | Brief-step reasoning instructions | 7/7 | 100.0% | 4.88 | 29 |
| 04 | Zero-shot instructions | 6/7 | 85.7% | 4.85 | 27 |
| 05 | Persona instructions in message | 6/7 | 85.7% | 4.75 | 28 |
| 06 | Temperature 0.2 | 7/7 | 100.0% | 4.43 | 29 |
| 07 | Temperature 0.9 | 6/7 | 85.7% | 4.52 | 28 |
| 08 | Top-p 0.65 | 6/7 | 85.7% | 4.52 | 28 |
| 09 | Frequency penalty 0.5 | 7/7 | 100.0% | 4.49 | 29 |
| 10 | Presence penalty 0.5 | 7/7 | 100.0% | 4.44 | 29 |
| 11 | Max completion tokens 250 | 7/7 | 100.0% | 4.55 | 29 |
| 12 | Compact instructions | 6/7 | 85.7% | 4.35 | 28 |

**Interpretation:** Condition 0 (Friendly Advisor, system instructions, few-shot, direct guidance) obtained 7/7. Brief-step instructions, temperature 0.2, frequency penalty 0.5, presence penalty 0.5, and a 250-token output cap also obtained 7/7. Zero-shot, in-message instructions, higher temperature, top-p 0.65, and compact instructions each obtained 6/7 because of the same policy-follow-up case. The Expert and Cautious personas obtained 6/7. Because there is only one run per matched scenario and several conditions tie, these outcomes do **not** demonstrate statistically reliable superiority of any particular parameter.

#### 4. Representative ReAct observations

**Successful fact lookup:** For a question comparing four single lessons with the package and asking about Sunday at 8 PM, the agent used `lookup_business_info` and correctly calculated $100 versus $90, a $10 saving. It rejected Sunday because CedarBridge is closed that day. The final answer did not assert a confirmed booking.

**Lead authorization:** When a user provided a name, email and tutoring need without explicit consent, the agent requested permission and made no write call. With explicit consent, the model requested `record_customer_interest`, and the validator returned `ok` in dry-run mode. This corrects the original version's brittle consent validation without requiring the customer to recite an exact phrase.

**Feedback tool selection:** On the undocumented 35% scholarship question, six conditions chose `lookup_business_info → record_feedback → final_answer`; seven chose `lookup_business_info → final_answer`. In the seven failed cases, the grounding audit recorded `coverage=resolved` and `independent_coverage=resolved`, even though the sources did not establish whether the scholarship exists. The two semantic checks thus exhibited the **same failure mode**, rather than reliably correcting it.

**Simulation and response-quality limitation:** In several successful write-tool tests, the final answer was only a simulation acknowledgment, e.g., "Your feedback was validated in simulation only; it was not stored or sent to staff." That is honest about `dry_run=True`, but in the scholarship scenario it omits the substantive answer: the business documents do not confirm the scholarship. **A tool-policy PASS is not equivalent to a fully helpful response.**

#### 5. Persona and prompting comparison

**Friendly Advisor:** The selected default for student-facing interactions. In normal factual responses it offered clear, conversational follow-up questions; its final tool-policy score was 7/7. The Strict Academic Expert produced more concise, fact-oriented answers and cited the business summary, while the Cautious Helper emphasized careful permission and follow-up. Because the three human-rating columns in the CSV are blank, describing Friendly as *most natural* is an **informed qualitative preference**, not a measured human preference or validated numerical score.

**Reasoning instructions:** In the 91-run comparison, direct and `brief_steps` prompting both achieved 7/7 tool-policy checks for the Friendly persona. The `brief_steps` mode prompts a short, user-visible action plan, rather than explicitly intervening on private reasoning. A **separate controlled direct-versus-structured CoT-style ablation** (Section 7) adds an explicit evidence-verification checklist to an otherwise unchanged prompt. Neither study reveals the model's private chain-of-thought.

**Examples and placement:** Few-shot/system achieved 7/7, zero-shot/system 6/7, and few-shot/in-message 6/7. The difference is localized to recording the undocumented policy question; remaining answers were similar. This is limited evidence favoring the default configuration, not a proof that few-shot or system placement is generally superior.

**Generation parameters:** Lowering temperature to 0.2 retained 7/7; raising it to 0.9 returned 6/7. The top-p, penalties and output-token variants generally preserved factual-service answers, with some differences in recording the unanswered policy case. Differences among 6/7 and 7/7 in a seven-item, single-round experiment are too small to justify strong claims about decoding configurations.

**Instructions:** Full instructions (7/7) vs compact instructions (6/7) suggests the longer operational guidance may have helped with the unresolved-policy workflow in this instance. The result is limited by sample size and stochastic variation.

#### 6. Comparison with the earlier committed experiment

The numbers below compare the **two 91-run implementation-stage CSVs**, which are separate from the 14-run reasoning-prompt ablation.

| Committed experiment file | Tool-policy passes | Mean scenario latency | Model requests |
|---|---:|---:|---:|
| `experiment_outputs/standard.csv` (earlier implementation) | 77/91 (84.6%) | 6.45 s | 305 |
| `experiment_outputs/final_standard.csv` (submitted implementation) | 84/91 (92.3%) | 4.60 s | 370 |

The submitted version recorded **7 additional matches**, an increase of **7.7 percentage points**, while making 65 more model requests (about 21% more). Its observed mean latency was **1.85 seconds lower** in this run. Latency is influenced by network and service conditions, so the difference should not be attributed to a particular code change. The architecture and prompts changed between runs; this is a **development-stage comparison**, not an isolated causal experiment. Dollar spending was not recorded, so no measured API cost is asserted.

##### 6.1 Qualitative observations from representative outputs

The stored final answers provide a useful *qualitative* cross-check, but no human 1–5 scores were entered in the evaluation CSV:

| Matched situation | Friendly Advisor | Strict Academic Expert | Cautious Helper |
|---|---|---|---|
| Four-lesson pricing and Sunday hours | Correct arithmetic and clear follow-up invitation | Correct arithmetic with more formal source-oriented wording | Correct arithmetic with cautious scheduling language |
| Inquiry without storage consent | Requests scoped permission instead of writing a lead | Requests explicit authorization | Requests explicit authorization |
| Undocumented scholarship | Invokes feedback in the baseline condition, but its simulation acknowledgment does not fully answer the question | States that scholarship is undocumented but fails to log the unanswered question | Same unresolved feedback-logging failure |

These are **reviewer observations**, not human rating measurements. Tool-policy compliance and customer-facing answer completeness should be reported separately. A grader can inspect the corresponding `answer` and `trace_json` entries in `final_standard.csv` to reproduce the interpretations.

#### 7. Direct versus structured CoT-style prompting

The separate evaluation in `experiment_outputs/cot_ablation.csv` contains **14 completed runs without runtime errors**: seven matched business scenarios tested once with the existing direct prompt and once with an otherwise identical prompt augmented by the checklist in `run_cot_ablation.py`. The variant order alternates by scenario. Both use the Friendly Advisor, `gpt-4.1-mini`, persona-default settings, a few-shot example and system-level placement. `agent_core.py` is unchanged, and write actions remain in `dry_run=True` mode.

The added prompt asks the model to identify the real question, look for supporting sources or tool observations, check arithmetic and consent/action constraints, and choose a supported action or final answer. It requests a short optional `Decision basis:` sentence about observable evidence. **It does not expose or measure private internal chain-of-thought**; the experiment tests the behavioral effect of an explicit CoT-style reasoning instruction.

| Metric | Direct | Structured CoT-style |
|---|---:|---:|
| Completed runs | 7/7 | 7/7 |
| Tool-policy passes | **7/7** | **7/7** |
| Mean latency | 7.66 s | 4.74 s |
| Median latency | 4.43 s | 4.42 s |
| Mean model requests per run | 4.14 | 4.14 |
| Mean tool observations per run | 1.00 | 1.00 |
| Answers containing `Decision basis:` | 0/7 | 2/7 |
| Runtime/API errors | 0 | 0 |

The tool-policy results were identical for all seven scenarios. Both variants performed the pricing arithmetic correctly ($100 for four separate lessons, $90 for the package, $10 saved), observed Sunday closure, withheld lead storage without consent, validated the consented lead in simulation, recorded complaints and the unanswered scholarship question in simulation, and answered the Arabic service question in Arabic.

The structured variant included an explicit evidence-basis label for the pricing and unsupported-subject responses. It did not consistently produce that label elsewhere; `dry_run` response replacement is one reason the requested presentation can be absent. On the scholarship scenario, both variants called the correct logging tool; only the structured final reply explicitly said the documents did not confirm the policy before acknowledging the simulated feedback. This illustrates a potential answer-completeness gain in one case, **not** a scored helpfulness improvement.

The direct condition contained a 25.95-second pricing run. It inflates the direct mean relative to the structured mean, while the medians (4.43 s vs 4.42 s) were nearly identical. Accordingly, the results **do not establish lower latency or improved tool correctness from CoT prompting**. The small single-round sample, blank human-rating fields, and fixed model limit broader conclusions.

#### 8. Evaluation limitations and future work

1. The primary score measures tool-policy behavior only; factuality, usefulness and naturalness must be evaluated from answers. The stored `human_*` fields are blank, so no fabricated human evaluation figures are included.
2. The source-grounding audits can agree incorrectly: both labeled the missing scholarship policy "resolved" in failed cases. Agreement between LLM verifiers is not proof of correctness. The `lookup_business_info` tool description also says it retrieves short relevant excerpts, whereas this version supplies the entire approved source texts; this documented mismatch remains in the frozen evaluated code.
3. Some `dry_run` response rewrites prioritized avoiding false claims about data persistence but failed to answer the user's substantive question. Real application behavior and simulated evaluation mode should be distinguished.
4. Each experimental comparison uses a small set of independent one-turn scenarios, with one observation per matched question/configuration. The 14-run CoT-style prompt ablation does not measure hidden reasoning, and its latency means are sensitive to an outlier. Multi-turn consent and unforeseen policies require separate future evaluation.
5. Model type was held fixed to isolate settings. The assignment's "model type" dimension can be acknowledged as a possible extension rather than claiming it was tested.

#### 9. Conclusion

The agent implements a manual, traceable ReAct workflow with configurable personas and controlled prompt/model-parameter comparisons. The final tool-policy pass rate was 84/91 (92.3%), compared with 77/91 (84.6%) in the earlier implementation-stage CSV, with strong handling of documented service questions, complaints and explicit lead consent in these tests. A separate matched CoT-style prompt evaluation achieved 7/7 tool-policy checks under both direct and structured conditions, without establishing a measurable advantage in tool correctness. The persistent failure concerned detecting that an undocumented policy question still required follow-up, including when independent LLM audits agreed on an incorrect resolution. The principal lesson is that **tool validation, grounded uncertainty management and honest execution reporting matter at least as much as persona styling or generation settings**. The results are suitable for course-level evaluation when their scope and limitations are reported transparently.


In [11]:
# Read two representative ReAct traces from the committed final experiment CSV.
import json
if results_path.exists():
    for condition, scenario in [('0', 'unknown_policy'), ('1', 'unknown_policy')]:
        row = next((r for r in records if r['condition'] == condition and r['scenario'] == scenario), None)
        if row:
            print(f"\nCondition {condition} / {scenario} — proxy_pass={row['proxy_pass']}")
            for step in json.loads(row.get('trace_json') or '[]'):
                print('  ', step.get('step'), '|',
                      step.get('tool') or step.get('decision') or '', '|',
                      step.get('status') or '', '|',
                      step.get('requested_tools') or '')
            print('  Final answer:', row.get('answer', '')[:350])



Condition 0 / unknown_policy — proxy_pass=True
   reason | tool_request |  | ['lookup_business_info']
   act_observe | lookup_business_info | ok | 
   reason | tool_request |  | ['record_feedback']
   act_observe | record_feedback | ok | 
   reason | final_answer |  | 
   grounding_audit |  | ok | 
  Final answer: Your feedback was validated in simulation only; it was not stored or sent to staff.

Condition 1 / unknown_policy — proxy_pass=False
   reason | tool_request |  | ['lookup_business_info']
   act_observe | lookup_business_info | ok | 
   reason | final_answer |  | 
   grounding_audit |  | ok | 
  Final answer: CedarBridge Learning's official information does not mention any scholarship or discount policy, including a 35% scholarship for Grade 12 students. The pricing is fixed as listed: $25 for an individual 60-minute lesson, $15 per student for small-group lessons, and $90 for a package of four individual lessons. No additional discounts or scholarships


## 12. Reflection — five required questions

### Chapter 4 — Reflection (Final Frozen Agent)

**Evidence:** 91 matched runs from `final_standard.csv` (13 conditions × 7 scenarios), the earlier 91-run `standard.csv`, and a separate 14-run direct-versus-structured CoT-style evaluation in `cot_ablation.csv`. Metrics refer to automated *tool-policy compliance*; human factuality/helpfulness/naturalness ratings have not been supplied.

#### 1. Which persona gave the most helpful or natural results?

For a student-facing tutoring assistant, I would select **Noura — Friendly Advisor**. In the inspected responses, it explained service policies in accessible language and frequently followed with a practical invitation to continue, which suits students and parents better than a purely formal answer. It passed **7/7** tool-policy checks. The **Strict Academic Expert** was useful for concise, source-oriented pricing and eligibility explanations, while the **Cautious Helper** communicated a more measured tone for customer details and follow-up; each passed **6/7**. Their single failure was the undocumented scholarship question, where neither invoked `record_feedback`. This is my qualitative preference based on the stored responses, not a claim of statistically measured naturalness: the CSV's three human-rating fields remain empty. The dry-run confirmation messages are also less natural than normal customer-facing replies.

#### 2. Which prompt and configuration combination performed best for your use case?

The best **practical default** was Friendly Advisor with direct instructions, a few-shot worked example, and persona instructions in the system message, using its configured temperature of **0.6** and `gpt-4.1-mini`. This baseline achieved **7/7** on the seven tool-policy scenarios. It tied with brief-step guidance, temperature 0.2, frequency penalty 0.5, presence penalty 0.5, and a 250-token completion limit. In contrast, zero-shot, in-message instructions, temperature 0.9, top-p 0.65, and compact instructions each obtained 6/7 because the agent skipped recording the undocumented scholarship question. I prefer the baseline over tied alternatives because its prompt structure is simple and its conversational wording is appropriate for the target audience. The experiment used one run per condition and seven cases, so I cannot conclude that a particular hyperparameter is consistently superior. The 91-run `brief_steps` comparison tested concise, visible planning instructions. To evaluate the requested thinking-mechanism contrast more directly, I also ran a separate **direct vs. structured CoT-style prompt ablation** with all other settings fixed. Both obtained **7/7 tool-policy passes** across the same seven questions. I would therefore retain the direct prompt as the practical default: the checklist did not improve this metric, and the evaluation is too small to establish an overall winner. The structured prompt sometimes supplied a useful short evidence basis, but this experiment does not expose private chain-of-thought.

#### 3. How well did your agent reason and use tools?

The manual LangGraph ReAct cycle worked as intended: a reasoning node selected whether to answer or request a tool; my Python code executed the selected action and returned an observation; the next model step formed the response. The final run completed **91/91** scenarios with no API/runtime errors and achieved **84/91 (92.3%)** automated tool-policy matches. All 13 multi-step pricing cases calculated $100 for four separate lessons, $90 for the package and $10 savings, while rejecting the Sunday 8 PM request. The agent withheld lead storage in all 13 no-consent cases and successfully validated the lead-writing tool in all 13 explicit-consent cases. Complaints were handled by `record_feedback` in all conditions. The main weakness was unanswered-policy escalation: only **6/13** conditions recorded the unknown scholarship question. Importantly, passing the tool-use proxy did not guarantee a fully informative answer. Some simulated write confirmations reported only that nothing had actually been stored, omitting the requested policy explanation. In the separate CoT-style ablation, both direct and structured prompting passed **7/7** tool-policy checks, with the same mean **4.14 model calls** and **1.00 tool calls** per case. The structured variant explicitly noted missing scholarship evidence in its final reply, while the direct variant returned only the simulation acknowledgment; this is a qualitative difference in one matched example, not a measured overall gain.

#### 4. Did your instructions block change the agent's behavior noticeably? How?

Yes, though not uniformly. The full operational instructions separated the agent's persona from concrete requirements: cite official information, avoid inventing policies or bookings, request scoped permission before storing contact data, record complaints/unanswered questions, and interpret tool observations. On the matched final test set, **full instructions passed 7/7** versus **6/7** with compact instructions; **few-shot/system placement passed 7/7**, while zero-shot/system and few-shot/in-message each passed **6/7**. Those small differences arose from whether `record_feedback` was called for an undocumented policy, not from errors in basic pricing or subject eligibility. Comparing the **two CSVs committed to the repository**, the policy-match rate rose from **77/91 (84.6%)** in `standard.csv` to **84/91 (92.3%)** in `final_standard.csv`—seven more matches. Because multiple implementation components changed, this cannot be attributed exclusively to prompt changes. The experiment suggests instructions help express operational intent but do not guarantee that the model or its self-checker applies it correctly. In the separate controlled reasoning-prompt comparison, adding a CoT-style evidence checklist did **not** change tool-policy success (7/7 in each condition). The requested `Decision basis:` appeared in **2/7** structured responses versus **0/7** direct responses, showing some visible formatting change but not evidence of a higher overall success rate.

#### 5. What were the biggest implementation challenges?

The hardest issue was **the gap between natural-language interpretation and safe tool execution**. My earlier consent validator accepted only narrow wording, rejecting a valid sentence that explicitly granted permission. Instead of adding more phrases to a whitelist, I redesigned consent interpretation to consider the conversation and scope of permission, while retaining deterministic checks for email format and whether a write could occur. The final experiment passed all explicit-consent scenarios, although the one-turn test set does not establish performance on every informal multi-turn reply. The second challenge was **distinguishing missing source evidence from a definite negative policy**. Several personas answered that no scholarship was documented but failed to log the unresolved question. Even when I added semantic source-coverage verification, both checks sometimes labeled the question as resolved; this showed that agreement among LLM judges is not necessarily reliable. A third challenge was **truthful side-effect reporting**: in dry-run experiments, a successful tool validation must not be described as a real customer record or guaranteed staff follow-up. Additional verification contributed to **370 model calls** in the final run compared with **305** in the preceding committed `standard.csv`. The recorded mean latency, however, decreased from **6.45 s** to **4.60 s** across those runs; network and execution conditions were not controlled sufficiently to attribute latency changes to a single design decision. The additional reasoning-prompt experiment reinforced the need to distinguish an explicit thinking instruction from verified improvements: both conditions passed all seven tool-policy checks, and a single long direct run (25.95 s) distorted the average latency comparison even though the medians were almost identical. Overall, this assignment showed me that building an agent is not only about generating convincing text: it requires traceable actions, evidence-based uncertainty, privacy-preserving validation, and an evaluation that separates operational success from response quality.
